# Optimizing Apache Spark

{doc}`Spark and Databricks <01_spark_and_databricks>` showed how to find out
what a query did, and
{doc}`Using Apache Spark with Delta Lake <02_using_apache_spark_with_delta_lake>`
how tables are stored. This notebook puts both to work on the question
every data engineer eventually faces: **how do I make this faster and
cheaper?**

Almost all of a Spark query's time goes to a few things: reading data from
storage, moving it between machines in shuffles, and waiting for the slowest
task of a stage. The techniques below are ordered by how much they usually
help, starting with the biggest win: reading less data. They build on the
concepts in
{doc}`Distributed Systems Programming Fundamentals <../01_introduction_to_apache_spark/03_distributed_systems_programming_fundamentals/03_distributed_systems_programming_fundamentals>`.

The examples use `samples.tpch.lineitem`, a table of 30 million order lines
from the TPC-H benchmark, which is large enough for layout to matter.

In [1]:
from pyspark.sql import functions as F

from project.utils.sparksession import get_spark_session

spark = get_spark_session(profile="free-edition")

for table in ["optimize03_lineitem", "optimize03_lineitem_clustered"]:
    spark.sql(f"DROP TABLE IF EXISTS workspace.default.{table}")

lineitem = spark.read.table("samples.tpch.lineitem")
lineitem.count()

29999795

## Read Less: Column Pruning and Predicate Pushdown

Delta tables store their data in Parquet files
([What is Delta Lake?](https://docs.databricks.com/aws/en/delta/)), a
**columnar** format: each column of a file is stored separately, so a query
that needs three columns reads only those three. Spark also pushes filters
down into the scan, so rows that do not match are discarded as early as
possible. The plan shows both:

In [2]:
(
    lineitem
    .filter(F.col("l_shipdate") == "1995-01-01")
    .select("l_orderkey", "l_quantity", "l_extendedprice")
    .explain()
)

== Physical Plan ==
PhotonResultStage
+- PhotonColumnarToRow
   +- PhotonProject [l_orderkey#42294L, l_quantity#42298, l_extendedprice#42299]
      +- PhotonScan parquet samples.tpch.lineitem[l_orderkey#42294L,l_quantity#42298,l_extendedprice#42299,l_shipdate#42304] DataFilters: [isnotnull(l_shipdate#42304), (l_shipdate#42304 = 1995-01-01)], DictionaryFilters: [(l_shipdate#42304 = 1995-01-01)], Format: parquet, Location: PreparedDeltaFileIndex(1 paths)[s3://system-tables-prod-us-east-2-uc-metastore-bucket/metastore/b..., OptionalDataFilters: [], PartitionFilters: [], ReadSchema: struct<l_orderkey:bigint,l_quantity:decimal(18,2),l_extendedprice:decimal(18,2),l_shipdate:date>, RequiredDataFilters: [isnotnull(l_shipdate#42304), (l_shipdate#42304 = 1995-01-01)]


== Photon Explanation ==
The query is fully supported by Photon.
== Optimizer Statistics (table names per statistics state) ==
  missing = lineitem
  partial = 
  full    = 
Corrective actions: consider running the following comma

In the `PhotonScan` line, `ReadSchema` lists only the four columns the query
uses, out of sixteen, and `DataFilters` contains the `l_shipdate` condition.
You get this for free as long as you do not defeat it: select the columns
you need early rather than carrying `*` through a pipeline, and filter on
plain columns. The file statistics described next are kept per column, so a
direct range condition such as
`l_shipdate BETWEEN '1995-01-01' AND '1995-12-31'` is the form they can be
compared against most directly; prefer it to wrapping the column in a
function such as `year(l_shipdate) = 1995`.

## Read Less: Data Skipping

Pruning columns and filtering rows still means opening every file. Delta
goes further. Statistics for each data file are collected automatically on
write, and at query time Databricks uses them to **skip files** that aren't
relevant to the query
([Data skipping](https://docs.databricks.com/aws/en/delta/data-skipping)).

How many files a filter can skip depends on how the rows are laid out. To
measure it, the helper below computes the minimum and maximum of a column
for every file of a table, using the `_metadata.file_path` column, and
counts the files whose range contains the value: those are the files Spark
has to read.

In [3]:
def files_to_read(table: str, column: str, value: str) -> tuple[int, int]:
    """Return (total files, files whose min/max range contains `value`)."""
    per_file = (
        spark.read.table(table)
        .groupBy(F.col("_metadata.file_path").alias("file"))
        .agg(F.min(column).alias("min_value"), F.max(column).alias("max_value"))
    )
    total = per_file.count()
    must_read = per_file.filter((F.col("min_value") <= value) & (F.col("max_value") >= value)).count()
    return total, must_read

Make a copy of `lineitem` in the course schema. The copy keeps the order of
the source, which is sorted by order key, not by date:

In [4]:
spark.sql("""
    CREATE TABLE workspace.default.optimize03_lineitem
    AS SELECT * FROM samples.tpch.lineitem
""")

files_to_read("workspace.default.optimize03_lineitem", "l_shipdate", "1995-01-01")

(10, 10)

Every file contains shipments from the whole seven-year date range, so a
filter for one day has to read all of them. Data skipping cannot help.

Now create a second copy with **liquid clustering** on `l_shipdate`, which
was introduced in
{doc}`Using Apache Spark with Delta Lake <02_using_apache_spark_with_delta_lake>`.
Liquid clustering groups data by the clustering keys, and Databricks
recommends it for all new tables
([Use liquid clustering for tables](https://docs.databricks.com/aws/en/tables/clustering)).
A smaller target file size, set with the `delta.targetFileSize` table
property
([Control data file size](https://docs.databricks.com/aws/en/tables/tune-file-size#set-a-target-file-size)),
gives the clustering more files to spread the dates over, which suits a
table of this size:

In [5]:
spark.sql("""
    CREATE TABLE workspace.default.optimize03_lineitem_clustered
    CLUSTER BY (l_shipdate)
    TBLPROPERTIES ('delta.targetFileSize' = '32mb')
    AS SELECT * FROM samples.tpch.lineitem
""")

files_to_read("workspace.default.optimize03_lineitem_clustered", "l_shipdate", "1995-01-01")

(32, 1)

:::::{dropdown} Example: How Much Does Clustering Save?
:open:
:color: primary
:icon: pencil
:class-body: example-body

A query sums the revenue shipped on 1 January 1995. How much of each table
does it have to read?

**Unclustered copy.** All 10 files have a date range that includes
1 January 1995, so Spark reads

$$
\frac{10}{10} = 100\%
$$

of the files.

**Clustered copy.** The data is spread over 32 files, each
covering a narrow range of dates, and only 1 of them can
contain the day:

$$
\frac{1}{32} \approx 3.1\%
$$

Spark skips the other files without opening them. Both tables return the
same answer; the clustered one simply reads a small fraction of the data to
get it. The saving grows with the table: on a table of thousands of files,
a selective filter on a clustering column can skip almost all of them.
:::::

Both queries return the same result:

In [6]:
for table in ["optimize03_lineitem", "optimize03_lineitem_clustered"]:
    revenue = (
        spark.read.table(f"workspace.default.{table}")
        .filter(F.col("l_shipdate") == "1995-01-01")
        .agg(F.sum("l_extendedprice").alias("revenue"))
        .first()["revenue"]
    )
    print(table, revenue)

optimize03_lineitem 472057538.40


optimize03_lineitem_clustered 472057538.40


On data this small the difference in run time is a fraction of a second and
varies from run to run, which is why this notebook measures files rather
than seconds. To see the effect on a real workload, compare the **files
pruned** and **bytes read** in the query profile, or the `pruned_files` and
`read_bytes` columns of `system.query.history`, introduced in
{doc}`Spark and Databricks <01_spark_and_databricks>`.

### Choosing Clustering Columns

Cluster by the columns most frequently used in query filters, typically a
date or timestamp and perhaps one or two frequently filtered keys, such as a
customer or store ID. You can specify up to four clustering keys, and
`CLUSTER BY AUTO` lets Databricks choose them from the table's historical
query workload
([Use liquid clustering for tables](https://docs.databricks.com/aws/en/tables/clustering#key-selection-guidelines)).

Liquid clustering replaces two older techniques you will still meet in
existing tables
([Use liquid clustering for tables](https://docs.databricks.com/aws/en/tables/clustering)):

- **Partitioning** (`partitionBy` / `PARTITIONED BY`) splits a table into
  one folder per value of a column. It works well only for low or known
  cardinality columns, and Databricks advises not to partition tables under
  1 TB
  ([When to partition tables on Databricks](https://docs.databricks.com/aws/en/tables/partitions)).
  Unlike clustering keys, partitioning is rigid and difficult to alter.
- **Z-ordering** (`OPTIMIZE ... ZORDER BY`) co-locates related values in
  the same files, and is not compatible with liquid clustering.

For new tables on Databricks, use liquid clustering.

## Right-Size Files

{doc}`Using Apache Spark with Delta Lake <02_using_apache_spark_with_delta_lake>`
showed a table of 200 small files compacted into one by `OPTIMIZE`. Both
extremes hurt: too many small files waste time on opening files and reading
metadata, and too few huge files limit how many tasks can read in parallel
and how many files can be skipped. On Databricks, Unity Catalog managed tables
have their file sizes tuned automatically, optimized writes and auto
compaction are always on for `MERGE`, `UPDATE` and `DELETE`, and the target
file size is autotuned from the table size: 256 MB for tables under 2.56 TB
([Control data file size](https://docs.databricks.com/aws/en/tables/tune-file-size)).
**Predictive optimization** runs `OPTIMIZE` on managed tables for you
([Predictive optimization](https://docs.databricks.com/aws/en/optimizations/predictive-optimization)).

What remains in your control is avoiding the patterns that create small
files in the first place: very frequent tiny appends, and `repartition`
with a large number before a write. As explained in
{ref}`Controlling Partitioning <repartition-vs-coalesce>`, use `coalesce`
to reduce the number of output files.

## Shuffle Less: Joins

After reading, shuffles are the next big cost. Joins are the most common
source of them, and the strategy Spark picks matters, as described in
{ref}`Join Strategies <join-strategies>`. Joining `lineitem` with the
50,000-row `supplier` table:

In [7]:
supplier = spark.read.table("samples.tpch.supplier")

(
    lineitem
    .join(supplier, lineitem.l_suppkey == supplier.s_suppkey)
    .groupBy("s_nationkey")
    .agg(F.sum("l_extendedprice").alias("revenue"))
    .explain()
)

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonGroupingAgg(keys=[s_nationkey#44176L], functions=[finalmerge_sum(merge sum#44205, isEmpty#44206) AS sum(l_extendedprice)#44184])
         +- PhotonShuffleExchangeSource false
            +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#39136]
               +- PhotonShuffleExchangeSink hashpartitioning(s_nationkey#44176L, 1024)
                  +- PhotonGroupingAgg(keys=[s_nationkey#44176L], functions=[partial_sum(l_extendedprice#44162) AS (sum#44205, isEmpty#44206)])
                     +- PhotonProject [l_extendedprice#44162, s_nationkey#44176L]
                        +- PhotonBroadcastHashJoin [l_suppkey#44159L], [s_suppkey#44173L], Inner, BuildRight, false, true, false
                           :- PhotonScan parquet samples.tpch.lineitem[l_suppkey#44159L,l_extendedprice#44162] DataFilters: [isnotnull(l_suppkey#44159L)], DictionaryFilters:

Spark chose a `PhotonBroadcastHashJoin` by itself: `supplier` is small, so a
copy is sent to every executor and `lineitem` is not shuffled for the join.
The only `hashpartitioning` exchange left is the one for the final
`groupBy` on `s_nationkey`.

Joining with the 7.5-million-row `orders` table instead:

In [8]:
orders = spark.read.table("samples.tpch.orders")

(
    lineitem
    .join(orders, lineitem.l_orderkey == orders.o_orderkey)
    .groupBy("o_orderpriority")
    .agg(F.sum("l_extendedprice").alias("revenue"))
    .explain()
)

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonGroupingAgg(keys=[o_orderpriority#44266], functions=[finalmerge_sum(merge sum#44295, isEmpty#44296) AS sum(l_extendedprice)#44274])
         +- PhotonShuffleExchangeSource false
            +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#39305]
               +- PhotonShuffleExchangeSink hashpartitioning(o_orderpriority#44266, 1024)
                  +- PhotonGroupingAgg(keys=[o_orderpriority#44266], functions=[partial_sum(l_extendedprice#44250) AS (sum#44295, isEmpty#44296)])
                     +- PhotonProject [l_extendedprice#44250, o_orderpriority#44266]
                        +- PhotonShuffledHashJoin [l_orderkey#44245L], [o_orderkey#44261L], Inner, BuildRight
                           :- PhotonShuffleExchangeSource false
                           :  +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#39289]
                           :

`orders` is too large to broadcast, so Spark uses a `PhotonShuffledHashJoin`,
with a `hashpartitioning` exchange on the join key for **both** sides: all
30 million `lineitem` rows and all 7.5 million `orders` rows cross the
network. Ways to make such a join cheaper:

- **Filter and project first.** Only rows and columns that survive into the
  join need shuffling. Spark pushes simple filters down automatically, but
  not filters that are only known to be selective after another join.
- **Aggregate before joining.** If you only need one row per key from a
  large table, `groupBy` it before the join, so the shuffle moves the
  aggregate rather than every detail row.
- **Broadcast when you know better.** Spark decides on broadcasting from
  size estimates. If a filtered table is much smaller than its estimate,
  `F.broadcast(df)` forces the broadcast regardless of the threshold
  ([Hints](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-qry-select-hints#join-hint-types)).
  Keep broadcast tables small: every executor receives a full copy.
- **Keep statistics current.** `ANALYZE TABLE ... COMPUTE STATISTICS FOR ALL
  COLUMNS` collects table and column statistics, which the query optimizer
  uses to generate an optimal plan; predictive optimization runs `ANALYZE`
  on managed tables
  ([ANALYZE TABLE](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-aux-analyze-table)).
  The `Optimizer Statistics` section at the bottom of every plan above shows
  which tables lack statistics.

Adaptive Query Execution, covered in
{ref}`Adaptive Query Execution <adaptive-query-execution>`, corrects some
of these decisions at run time, for example by switching to a broadcast join
when one side turns out to be small after filtering
([Adaptive query execution](https://docs.databricks.com/aws/en/optimizations/aqe#capabilities)).
On serverless compute it can't be turned off, as
{doc}`Spark and Databricks <01_spark_and_databricks>` showed.

## Handle Skew

A stage finishes when its slowest task finishes. If one key has far more
rows than the others, the task that gets that key does most of the work
while the rest of the cluster waits. To see what skew looks like, generate
two million rows where 60% share the same key:

In [9]:
events = spark.range(2_000_000).withColumn(
    "key",
    F.when(F.col("id") % 10 < 6, F.lit(0)).otherwise(F.col("id") % 1000),
)

rows_per_key = events.groupBy("key").count()

rows_per_key.agg(
    F.count("*").alias("keys"),
    F.max("count").alias("largest_key"),
    F.median("count").alias("median_key"),
).show()

+----+-----------+----------+
|keys|largest_key|median_key|
+----+-----------+----------+
| 401|    1200000|    2000.0|
+----+-----------+----------+



The largest key holds 1,200,000 rows, 600 times as many as the
median key. In a join or aggregation on `key`, one task would process
more than half of all the data.

Measuring rows per key like this, and comparing the maximum with the median,
is the quickest way to confirm skew. In the Spark UI it shows up as a stage
whose maximum task duration is far above the 75th percentile
([Skew and spill](https://docs.databricks.com/aws/en/optimizations/spark-ui-guide/long-spark-stage-page#skew)).

**Salting** spreads a hot key over several tasks: add a random number, the
salt, to the grouping key, aggregate by key and salt, then aggregate again
by key alone. The first aggregation is spread over many tasks, and the
second only has to combine 16 partial results per key:

In [10]:
SALT_BUCKETS = 16

salted = events.withColumn("salt", (F.rand(seed=42) * SALT_BUCKETS).cast("int"))

partial = salted.groupBy("key", "salt").count()
partial.agg(F.max("count").alias("largest_group_after_salting")).show()

totals = partial.groupBy("key").agg(F.sum("count").alias("count"))
totals.filter(F.col("key") == 0).show()

+---------------------------+
|largest_group_after_salting|
+---------------------------+
|                      75701|
+---------------------------+



+---+-------+
|key|  count|
+---+-------+
|  0|1200000|
+---+-------+



The largest group in the first, expensive aggregation shrank from
1,200,000 rows to 75,701, roughly
$1/16$ of the hot key, and the second aggregation restores the exact total.

For joins, Databricks handles skew by default through AQE
([Skew join optimization using skew hints](https://docs.databricks.com/aws/en/archive/legacy/skew-join)),
so try the simpler remedies from
{doc}`Distributed Systems Programming Fundamentals <../01_introduction_to_apache_spark/03_distributed_systems_programming_fundamentals/03_distributed_systems_programming_fundamentals>`
first, and salt only when a skewed aggregation or join is still the
bottleneck.

## Let the Engine Do the Work

Some of the biggest speed-ups come from staying inside what the engine can
optimise:

- **Photon.** Databricks' native vectorised engine replaces the JVM-based
  execution engine for the operations it supports, and is enabled on
  serverless compute. It doesn't support UDFs, RDD APIs or Dataset APIs;
  unsupported operations fall back to the Spark runtime
  ([What is Photon?](https://docs.databricks.com/aws/en/compute/photon)).
  The `Photon Explanation` section of every plan above says whether the
  whole query was supported.
- **Built-in functions over Python UDFs.** A Python UDF must serialise the
  data and move it out of the JVM to the Python interpreter. Expressing the
  same logic with `pyspark.sql.functions`, as in
  {doc}`DataFrame API Basics <../02_developing_applications_with_apache_spark/01_dataframe_api_basics>`,
  keeps it in the engine. If a UDF is unavoidable, pandas UDFs are up to 100x
  faster than Python UDFs because they use Apache Arrow
  ([User-defined functions (UDFs)](https://docs.databricks.com/aws/en/udf/#performance-considerations)).
- **Few actions.** Every action re-runs its plan from the source. A
  pipeline that calls `count()` for logging between steps reads its input
  several times. Use observed metrics, shown in
  {doc}`Spark and Databricks <01_spark_and_databricks>`, instead.

### Caching

On classic compute, `df.cache()` keeps a DataFrame in memory so that several
actions can reuse it. Serverless compute does not support the DataFrame and
SQL cache APIs
([Serverless compute limitations](https://docs.databricks.com/aws/en/compute/serverless/limitations#caching-limitations)):

In [11]:
lineitem.cache().count()

AnalysisException: [NOT_SUPPORTED_WITH_SERVERLESS] PERSIST TABLE is not supported on serverless compute. SQLSTATE: 0A000

JVM stacktrace:
org.apache.spark.sql.AnalysisException
	at com.databricks.serverless.ServerlessGCEdgeCheck$.throwError(ServerlessGCEdgeCheck.scala:72)
	at com.databricks.serverless.ServerlessGCEdgeCheck$.checkBlockCacheCommand(ServerlessGCEdgeCheck.scala:50)
	at org.apache.spark.sql.connect.planner.SparkConnectPlanner.handleCreateDataFrameSnapshotCommand(SparkConnectPlanner.scala:5700)
	at org.apache.spark.sql.connect.planner.SparkConnectPlanner.process(SparkConnectPlanner.scala:3936)
	at org.apache.spark.sql.connect.execution.ExecuteThreadRunner.$anonfun$handleCommand$1(ExecuteThreadRunner.scala:567)
	at scala.runtime.java8.JFunction0$mcV$sp.apply(JFunction0$mcV$sp.scala:18)
	at org.apache.spark.sql.execution.SQLExecution$.withPreExecutionPhaseWatch(SQLExecution.scala:1818)
	at org.apache.spark.sql.connect.execution.ExecuteThreadRunner.withPreExecutionPhaseWatch(Ex

The error is `NOT_SUPPORTED_WITH_SERVERLESS`. In practice this matters less
than it seems. The **disk cache** keeps
copies of remote Parquet data files on the nodes' local storage, and is
used automatically on first read where it is enabled
([Optimize performance with caching on Databricks](https://docs.databricks.com/aws/en/optimizations/disk-cache#disk-cache-vs-spark-cache)),
and SQL warehouses reuse the results of identical queries from their
**result cache** for up to 24 hours, until the underlying tables change
([Query caching](https://docs.databricks.com/aws/en/sql/user/queries/query-caching)). Where a pipeline really reuses an expensive
intermediate result, write it to a table. That also makes the pipeline
easier to debug and restart.

## Choosing Compute

Finally, the compute itself:

- **Serverless** is what Databricks recommends for most new workloads
  ([Compute configuration recommendations](https://docs.databricks.com/aws/en/compute/cluster-config-best-practices)).
  Serverless jobs can run in **performance-optimized** mode, which starts
  and runs faster for time-sensitive workloads, or in **standard** mode,
  which consumes fewer DBUs and suits workloads that can accept a start-up
  latency of about 4 to 6 minutes
  ([Run your Lakeflow Jobs with serverless compute](https://docs.databricks.com/aws/en/jobs/run-serverless-jobs#select-a-performance-mode)).
- **Classic** clusters are sized by you. If you see significant spill to
  disk or out-of-memory errors, give the instances more memory; Photon is
  most beneficial for SQL workloads and DataFrame operations with complex
  transformations
  ([Compute configuration recommendations](https://docs.databricks.com/aws/en/compute/cluster-config-best-practices)).

Whatever the compute, measure before and after every change: in the query
profile, in `system.query.history`, and in the cost tables shown in
{doc}`Spark and Databricks <01_spark_and_databricks>`.

## Common Mistakes

- **Partitioning a table that should be clustered.** Partitioning tables
  under 1 TB, or on high-cardinality columns, creates small files; use
  liquid clustering
  ([When to partition tables on Databricks](https://docs.databricks.com/aws/en/tables/partitions)).
- **Wrapping filter columns in functions.** Filter on the column itself so
  the per-column file statistics can be used.
- **Calling `cache()` on serverless compute.** It raises an error; write an
  expensive intermediate result to a table instead
  ([Serverless compute limitations](https://docs.databricks.com/aws/en/compute/serverless/limitations#caching-limitations)).
- **Salting before checking AQE.** Databricks already handles join skew
  through AQE
  ([Skew join optimization using skew hints](https://docs.databricks.com/aws/en/archive/legacy/skew-join)).
- **Judging an optimisation by one run time on small data.** Times vary from
  run to run; compare files and bytes read instead, as this page does.
- **Writing Python UDFs for logic built-in functions can express.** Photon
  doesn't run UDFs
  ([What is Photon?](https://docs.databricks.com/aws/en/compute/photon)).

## A Checklist

1. **Read less.** Select only the needed columns, filter early on plain
   columns, and cluster large tables by the columns queries filter on.
2. **Keep files healthy.** Avoid writing many small files, and let
   predictive optimization run `OPTIMIZE`, `VACUUM` and `ANALYZE`.
3. **Shuffle less.** Count the exchanges in the plan. Broadcast small
   tables, aggregate before joining, and keep statistics current.
4. **Check for skew.** Compare the largest key with the median, and the
   slowest task with the typical one.
5. **Stay in the engine.** Prefer built-in functions to Python UDFs, and
   avoid unnecessary actions.
6. **Measure.** Use query profiles and system tables to confirm that a
   change helped.

## Cleaning Up

In [12]:
for table in ["optimize03_lineitem", "optimize03_lineitem_clustered"]:
    spark.sql(f"DROP TABLE IF EXISTS workspace.default.{table}");

This completes the course material. To check what you have learned against
the exam, work through
{doc}`Hands-On Tasks Before the Exam <../05_exam_readiness/01_hands_on_tasks>`.

## Resources

- [Optimization recommendations on Databricks (Databricks)](https://docs.databricks.com/aws/en/optimizations/)
- [Use liquid clustering for tables (Databricks)](https://docs.databricks.com/aws/en/tables/clustering)
- [Data skipping for Delta Lake (Databricks)](https://docs.databricks.com/aws/en/delta/data-skipping)
- [Adaptive query execution (Databricks)](https://docs.databricks.com/aws/en/optimizations/aqe)
- [What is Photon? (Databricks)](https://docs.databricks.com/aws/en/compute/photon)
- [Performance Tuning (Apache Spark)](https://spark.apache.org/docs/latest/sql-performance-tuning.html)